# Feature Importance Analysis via Progressive Feature Removal

This notebook implements an iterative feature pruning sweep to analyze feature importance using Lexos' unified classification interface.

**Strategy:**
1. Initialize the `Classifier` with `features="all"` to automatically discover baseline corpus statistics features.
2. Extract the dynamically discovered feature list directly from the classifier instance.
3. Iteratively remove one feature at a time in a randomized sequence.
4. Track performance degradation or improvement metrics across each configuration by setting explicit feature subsets.

In [1]:
# Library imports
from pathlib import Path
import random
import numpy as np
import pandas as pd
from pathlib import Path


# Lexos components
from lexos.classification import Classifier, MLPPipeline
from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer
from lexos.io.loader import Loader

## Using Loader class to load the files

In [2]:
# Seed initialization and data directory resolution
SEED = 42
rng = random.Random(SEED)
np.random.seed(SEED)

# Setup cleaning pipelines
scrubber = Scrubber()
scrubber.add_pipe("lower_case")
scrubber.add_pipe("digits")
scrubber.add_pipe("punctuation")

tokenizer = Tokenizer(model="en_core_web_sm")

In [3]:
loader = Loader()
loader.reset() # just in case there is stuff in here

loader.load("../fed_papers")

print("Errors:", loader.errors)
print("Loaded Names:", loader.names)

Errors: []
Loaded Names: ['FED_18_C', 'FED_19_C', 'FED_20_C', 'FED_49_D', 'FED_50_D', 'FED_51_D', 'FED_52_D', 'FED_53_D', 'FED_54_D', 'FED_55_D', 'FED_56_D', 'FED_57_D', 'FED_58_D', 'FED_62_D', 'FED_63_D', 'FED_11_H', 'FED_12_H', 'FED_13_H', 'FED_15_H', 'FED_16_H', 'FED_17_H', 'FED_1_H', 'FED_21_H', 'FED_22_H', 'FED_23_H', 'FED_24_H', 'FED_25_H', 'FED_26_H', 'FED_27_H', 'FED_28_H', 'FED_29_H', 'FED_30_H', 'FED_31_H', 'FED_32_H', 'FED_33_H', 'FED_34_H', 'FED_35_H', 'FED_36_H', 'FED_59_H', 'FED_60_H', 'FED_61_H', 'FED_65_H', 'FED_66_H', 'FED_67_H', 'FED_68_H', 'FED_69_H', 'FED_6_H', 'FED_70_H', 'FED_71_H', 'FED_72_H', 'FED_73_H', 'FED_74_H', 'FED_75_H', 'FED_76_H', 'FED_77_H', 'FED_78_H', 'FED_79_H', 'FED_7_H', 'FED_80_H', 'FED_81_H', 'FED_82_H', 'FED_83_H', 'FED_84_H', 'FED_85_H', 'FED_8_H', 'FED_9_H', 'FED_2_J', 'FED_3_J', 'FED_4_J', 'FED_5_J', 'FED_64_J', 'FED_10_M', 'FED_14_M', 'FED_37_M', 'FED_38_M', 'FED_39_M', 'FED_40_M', 'FED_41_M', 'FED_42_M', 'FED_43_M', 'FED_44_M', 'FED_45_M',

In [4]:
# 1. Access the internal records dataframe directly from your loader object
df_all = loader.df

# 2. Extract training records using non-capturing regex groups on the "name" column
# (?:_H|_M)$ ensures it matches the ending without triggering pandas warnings
df_train = df_all[df_all["name"].str.contains(r"(?:_H|_M)$", case=False, regex=True, na=False)].copy()

# Map the author labels cleanly based on that trailing identifier flag
df_train["label"] = df_train["name"].apply(lambda name: "HAMILTON" if name.upper().endswith("_H") else "MADISON")

# 3. Extract unknown/disputed records cleanly 
df_unknown = df_all[df_all["name"].str.contains(r"(?:_D|_C)$", case=False, regex=True, na=False)].copy()

# 4. Extract data lists required for your downstream pipelines
train_texts = df_train["text"].tolist()
train_labels = df_train["label"].tolist()
train_ids = df_train["name"].tolist()

unknown_texts = df_unknown["text"].tolist()
unknown_ids = df_unknown["name"].tolist()

# Verify your parsing metrics
print(f"Training docs extracted from Loader: {len(train_texts)}")
print(f"Unknown docs extracted from Loader: {len(unknown_texts)}")
print("\nTraining Class Distribution:")
print(pd.Series(train_labels).value_counts())

/Users/jon/lexos/src/lexos/corpus/corpus_stats.py:385: UserWarning: Loaded spaCy model 'sent_ud_sm' does not include a tagger; skipping syllable-based features.
  warnings.warn(


Training docs extracted from Loader: 130
Unknown docs extracted from Loader: 30

Training Class Distribution:
HAMILTON    102
MADISON      28
Name: count, dtype: int64


In [5]:
df_all.shape[0]

170

In [ ]:
df_train.shape[0]

In [ ]:
# Define the baseline training configuration strategy profile
baseline_strategy = MLPPipeline(
    seed=SEED,
    min_df=2,
    test_size=0.2,
    cv_splits=5,
    include_bigrams=True, # TODO: Still need to add bigrasm to the input layer
    use_smote=True,
    mlp_kwargs={
        "hidden_layer_sizes": (64,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "max_iter": 1000,
    },
)

# Instantiate a temporary baseline classifier to dynamically extract available corpus stats features
baseline_discoverer = Classifier(
    train_data=train_texts,
    labels=train_labels,
    pipeline=baseline_strategy,
    features="all"
)

# Call the build method explicitly to get the full list for the experiment sweep
all_corpus_stat_features = baseline_discoverer.discover_features()
feature_removal_order = all_corpus_stat_features.copy()
rng.shuffle(feature_removal_order) # Still need to implement different algorithms

print(f"CorpusStats features dynamically discovered: {len(all_corpus_stat_features)}")
print("Random removal order:\n", ", ".join(feature_removal_order))

In [4]:
# Setup the strategy profile and request a randomized feature removal sweep
mlp_strategy = MLPPipeline(
    seed=SEED,
    feature_removal="random", # Needs to add more algorithms
    mlp_kwargs={"hidden_layer_sizes": (64,), "max_iter": 1000}
)

# Initialize the classifier with 'all' features
classifier = Classifier(
    train_data=train_texts,
    labels=train_labels,
    pipeline=mlp_strategy,
    features="all"
)

# Run the entire sweep
sweep_results_df = classifier.feature_importance_sweep()

# Display the final summary report directly
print(sweep_results_df)# TODO: add the measured metric

/Users/jon/lexos/src/lexos/corpus/corpus_stats.py:385: UserWarning: Loaded spaCy model 'sent_ud_sm' does not include a tagger; skipping syllable-based features.
  warnings.warn(


   configuration          removed_feature  features_remaining  \
0       baseline                 baseline                  40   
1      remove_01           hapax_legomena                  39   
2      remove_02              total_terms                  38   
3      remove_03        hapax_dislegomena                  37   
4      remove_04     hapax_legomenon_rate                  36   
5      remove_05              sconj_count                  35   
6      remove_06              propn_count                  34   
7      remove_07                  log_ttr                  33   
8      remove_08           sentence_count                  32   
9      remove_09               punc_count                  31   
10     remove_10            guiraud_index                  30   
11     remove_11               intj_count                  29   
12     remove_12                      ttr                  28   
13     remove_13               pron_count                  27   
14     remove_14         

In [ ]:
sweep_results_df

In [ ]:
# Assuming we are tracking holdout_macro_f1 as the primary evaluation metric
# Locate the row where the F1 score was at its absolute lowest
most_impactful_row = sweep_results_df.loc[sweep_results_df["holdout_macro_f1"].idxmin()]

print(f"Most Impactful Feature: {most_impactful_row['removed_feature']}")
print(f"F1 Score when removed: {most_impactful_row['holdout_macro_f1']:.4f}")

In [6]:
# Extract the baseline score from the first row
baseline_f1 = sweep_results_df.loc[sweep_results_df["configuration"] == "baseline", "holdout_macro_f1"].values[0]

# Calculate the drop for each step
sweep_results_df["f1_delta_vs_baseline"] = sweep_results_df["holdout_macro_f1"] - baseline_f1

# Sort the DataFrame to put the largest negative drops at the top
sorted_impact_df = sweep_results_df.sort_values(by="f1_delta_vs_baseline")

print(sorted_impact_df[["removed_feature", "features_remaining", "holdout_macro_f1", "f1_delta_vs_baseline"]])